## Restart policy

`spec.restartPolicy` controls what the kubelet does when a container exits. It applies to **all** containers in the pod — you can't set it per container.

| Policy | Behaviour |
|---|---|
| `Always` *(default for Pods)* | Restart every time it exits, regardless of exit code. Right for long-running services. |
| `OnFailure` | Restart only on a non-zero exit. Right for batch work that retries on failure, stops on success. |
| `Never` | Never restart. Right for one shot, success or failure. |

Three subtleties that matter:

- **Restart ≠ reschedule.** This is the *kubelet* restarting the container — same node, same pod, same volumes, just a new container process. If the **node** dies, only a higher-level controller (Deployment, ReplicaSet, DaemonSet) brings the pod back elsewhere. **A bare pod with `restartPolicy: Always` will not survive a node failure.**
- **CrashLoopBackOff.** When a container keeps crashing fast, the kubelet waits longer between restarts — 10s, 20s, 40s, up to 5 minutes. That shows in `kubectl get pods` as `CrashLoopBackOff`. It's not a phase — the pod is still `Running`; the *container* is `Waiting` with that reason.
- **Jobs default differently.** A Job's pod template defaults to `OnFailure`; the Job controller handles whole-pod failure separately (notebook 03).

The lesson to carry: `restartPolicy` heals a **crashed process**; it does nothing for a **lost node**. That's the exact gap Deployments fill — which is why almost nobody runs bare pods in production. On our map, restart policy is a property of the **Pods** box itself; the resurrection-across-nodes story belongs to the Workload kinds above it.